# Stage 7B — Put the prediction behind a local API

**Goal:** Send JSON to `/predict` and receive a probability, decision, and model version. Test valid and invalid requests.

**Why:** An API gives another program a stable way to use the model. Here we test it inside Colab with a local test client. It is not a public, always-on service.

**Before you start:** Run Stage 7A Cell 7 if you have not yet seen `Saved contract:`. This notebook starts from a fresh runtime.

In [ ]:
# Cell 2 — Why: install known versions of the API and test tools.
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "fastapi==0.141.1", "httpx==0.28.1"])
import fastapi, httpx
print("FastAPI:", fastapi.__version__, "HTTPX:", httpx.__version__)

In [ ]:
# Cell 3 — Why: load the saved model and its written contract.
from google.colab import drive
from pathlib import Path
import hashlib, json, joblib, platform, pandas as pd, numpy as np, sklearn

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
run_id = "20260926T120702352046Z"
run_dir = root / "runs" / run_id
record = json.loads((run_dir / "run.json").read_text())
model_file = run_dir / "model.joblib"
contract_file = root / "contracts" / run_id / "stage7a-v1.json"
if not contract_file.exists():
    raise ValueError("Run Stage 7A Cell 7 to save the input contract first")
contract = json.loads(contract_file.read_text())
if record["run_id"] != run_id or contract["model_run_id"] != run_id:
    raise ValueError("Model and contract versions differ")
if hashlib.sha256(model_file.read_bytes()).hexdigest() != record["model_sha256"]:
    raise ValueError("Saved model changed")
print("Verified model and contract:", run_id)

In [ ]:
# Cell 4 — Why: refuse an incompatible runtime before loading the model.
import joblib

current = {"python": platform.python_version(), "pandas": pd.__version__,
           "numpy": np.__version__, "scikit_learn": sklearn.__version__,
           "joblib": joblib.__version__}
if current != record["versions"]:
    raise RuntimeError(f"Model needs {record['versions']}; this runtime has {current}")
model = joblib.load(model_file)  # Our own hash-verified artifact.
known_types = set(model.named_steps["prepare"].named_transformers_["text"]
                  .named_steps["encode"].categories_[0])
print("Model loaded with matching versions")

In [ ]:
# Cell 5 — Why: turn one valid JSON request into training-style features.
from datetime import datetime

boroughs = {"BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND", "Unspecified"}

def make_row(item):
    if not item.complaint_type.strip() or len(item.complaint_type) > 200:
        raise ValueError("complaint_type must have 1–200 characters")
    if item.borough not in boroughs:
        raise ValueError("Unknown borough")
    if "T" not in item.created_date:
        raise ValueError("created_date needs a date and time")
    moment = datetime.fromisoformat(item.created_date)
    if moment.tzinfo is not None:
        raise ValueError("Use NYC local time without a timezone suffix")
    pd.Timestamp(moment).tz_localize("America/New_York", ambiguous="raise", nonexistent="raise")
    row = {"complaint_type": item.complaint_type, "borough": item.borough,
           "month": moment.month, "hour": moment.hour, "weekday": moment.weekday()}
    return pd.DataFrame([row], columns=record["features"])

print("Feature builder ready")

In [ ]:
# Cell 6 — Why: define the HTTP request contract and prediction endpoint.
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, StrictStr

class NewRequest(BaseModel):
    complaint_type: StrictStr
    borough: StrictStr
    created_date: StrictStr
    model_config = ConfigDict(extra="forbid")

app = FastAPI(title="NYC 311 72-hour prediction", version="stage7b-v1")

@app.post("/predict")
def predict(item: NewRequest):
    try:
        row = make_row(item)
    except Exception as error:
        raise HTTPException(status_code=422, detail=str(error)) from error
    probability = float(model.predict_proba(row)[0, 1])
    threshold = float(record["decision_threshold"])
    return {"probability_late": round(probability, 4),
            "predicted_label": int(probability >= threshold),
            "threshold": threshold, "model_run_id": run_id,
            "new_complaint_type": item.complaint_type not in known_types}

@app.get("/health")
def health():
    return {"status": "ready", "model_run_id": run_id}

print("Endpoints ready: /predict and /health")

In [ ]:
# Cell 7 — Why: send a real HTTP-style request without public hosting.
from fastapi.testclient import TestClient

client = TestClient(app)
sample = contract["example_request"]
response = client.post("/predict", json=sample)
assert response.status_code == 200, response.text
assert response.json()["model_run_id"] == run_id
assert 0 <= response.json()["probability_late"] <= 1
print("POST /predict:", response.status_code, response.json())
print("GET /health:", client.get("/health").json())

In [ ]:
# Cell 8 — Why: prove the API rejects missing, leaked, and invalid input.
bad = [
    {**sample, "closed_date": "2026-09-27T10:00:00"},
    {"borough": sample["borough"], "created_date": sample["created_date"]},
    {**sample, "created_date": "not a date"},
]
for item in bad:
    result = client.post("/predict", json=item)
    assert result.status_code == 422, result.text
    print("Rejected with HTTP", result.status_code)
print("All three invalid requests were rejected")

In [ ]:
# Cell 9 — Why: save the API test result with model and contract IDs.
from datetime import datetime, timezone

api_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
api_dir = root / "api_demos" / api_id
print("Creating API record folder...", flush=True)
api_dir.mkdir(parents=True, exist_ok=False)
import pydantic, starlette
result_record = {"api_id": api_id, "model_run_id": run_id,
                 "contract_version": contract["contract_version"],
                 "fastapi_version": fastapi.__version__, "httpx_version": httpx.__version__,
                 "pydantic_version": pydantic.__version__, "starlette_version": starlette.__version__,
                 "model_sha256": record["model_sha256"],
                 "contract_sha256": hashlib.sha256(contract_file.read_bytes()).hexdigest(),
                 "valid_status": response.status_code, "invalid_statuses": [422, 422, 422],
                 "example_response": response.json()}
print("Saving API test record...", flush=True)
(api_dir / "result.json").write_text(json.dumps(result_record, indent=2) + "\n", encoding="utf-8")
print("API demo ID:", api_id)
print("Saved API test result:", api_dir / "result.json")

# Stage 7B takeaway and quiz

**What you learned:** The API accepts a JSON request, validates it, calls the saved model, and returns a versioned response. A local test client lets us exercise the HTTP interface without keeping a Colab server running. We saved its test result and the exact model and input-contract IDs.

**Easy quiz:**

1. What does HTTP 200 mean for our valid request?
2. Why should an input with `closed_date` return HTTP 422?
3. What is the difference between `/health` and `/predict`?
4. Is this local Colab API available after the runtime shuts down?

Send me the output of Cells 7–9 and your answers. Stage 8 will investigate the lower November–December recall and design monitoring.